In [2]:
import polars as pl
import numpy as np
import json
from pathlib import Path

pl.enable_string_cache()

In [3]:
INPUT_PATH = Path("../data/processed/nyc_311_cleaned.csv")
OUTPUT_PATH = Path("../data/training/nyc_311_features.csv")
MAPPING_DIR = Path("../data/artifacts/mappings")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
MAPPING_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
df = pl.read_csv(INPUT_PATH, try_parse_dates=True)
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns}")

Shape: (1500000, 14)
Columns: ['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'location_type', 'borough', 'latitude', 'longitude', 'resolution_minutes', 'hour', 'weekday']


In [5]:
df = df.with_columns(
    pl.when(pl.col("resolution_minutes") < 30).then(0)
    .when(pl.col("resolution_minutes") < 60).then(1)
    .when(pl.col("resolution_minutes") < 90).then(2)
    .when(pl.col("resolution_minutes") < 180).then(3)
    .when(pl.col("resolution_minutes") < 360).then(4)
    .when(pl.col("resolution_minutes") < 1440).then(5)
    .when(pl.col("resolution_minutes") < 2880).then(6)
    .when(pl.col("resolution_minutes") < 5760).then(7)
    .when(pl.col("resolution_minutes") < 20160).then(8)
    .otherwise(9)
    .alias("target_bucket")
)

In [6]:
df = df.with_columns([
    pl.col("created_date").dt.hour().alias("hour"),
    pl.col("created_date").dt.minute().alias("minute"),
    pl.col("created_date").dt.second().alias("second"),
])

df = df.with_columns(
    ((pl.col("hour") * 3600) + (pl.col("minute") * 60) + pl.col("second")).alias("seconds_since_midnight")
)

df = df.with_columns(
    (pl.col("seconds_since_midnight") / 86400).alias("day_fraction")
)

df = df.with_columns([
    (2 * np.pi * pl.col("day_fraction")).sin().round(4).alias("time_sin"),
    (2 * np.pi * pl.col("day_fraction")).cos().round(4).alias("time_cos"),
])

In [7]:
df = df.with_columns(
    (pl.col("created_date").dt.weekday() - 1).alias("weekday")
)

df = df.with_columns(
    (pl.col("weekday") * 86400 + pl.col("seconds_since_midnight")).alias("week_seconds")
)

df = df.with_columns(
    (pl.col("week_seconds") / (7 * 86400)).alias("week_fraction")
)

df = df.with_columns([
    (2 * np.pi * pl.col("week_fraction")).sin().round(4).alias("week_sin"),
    (2 * np.pi * pl.col("week_fraction")).cos().round(4).alias("week_cos"),
])

df = df.with_columns(
    pl.when(pl.col("weekday").is_in([5, 6])).then(1).otherwise(0).alias("is_weekend")
)

In [8]:
latitude_min: float = df["latitude"].min() # type: ignore
latitude_max: float = df["latitude"].max() # type: ignore
longitude_min: float = df["longitude"].min() # type: ignore
longitude_max: float = df["longitude"].max() # type: ignore

df = df.with_columns([
    ((pl.col("latitude") - latitude_min) / (latitude_max - latitude_min)).alias("latitude_scaled"),
    ((pl.col("longitude") - longitude_min) / (longitude_max - longitude_min)).alias("longitude_scaled"),
])

scaler_metadata = {
    "latitude_min": latitude_min,
    "latitude_max": latitude_max,
    "longitude_min": longitude_min,
    "longitude_max": longitude_max,
}

with open(MAPPING_DIR / "scaler_metadata.json", "w") as f:
    json.dump(scaler_metadata, f, indent=2)

print("Scaler metadata saved.")
print(scaler_metadata)

Scaler metadata saved.
{'latitude_min': 40.49888624741073, 'latitude_max': 40.912868795316655, 'longitude_min': -74.25473797275497, 'longitude_max': -73.70036545591262}


In [9]:
df = df.with_columns(
    (pl.col("complaint_type") + "|" + pl.col("descriptor") + "|" + pl.col("location_type")).alias("combo")
)

print(f"Unique combos retained: {df['combo'].n_unique():,}")

Unique combos retained: 522


In [10]:
agency_vals = df["agency"].unique().sort().to_list()
agency_mapping = {}
agency_exprs = []

for val in agency_vals:
    key = f"agency_{val}"
    full_name = df.filter(pl.col("agency") == val)["agency_name"].unique().to_list()
    agency_mapping[key] = full_name[0] if full_name else val
    agency_exprs.append(pl.when(pl.col("agency") == val).then(1).otherwise(0).alias(key))

df = df.with_columns(agency_exprs).drop("agency")

with open(MAPPING_DIR / "agency_map.json", "w") as f:
    json.dump(agency_mapping, f, indent=2)

print(f"Agency: {len(agency_vals)} categories → {len(agency_exprs)} features")

Agency: 14 categories → 14 features


In [11]:
complaint_vals = df["complaint_type"].unique().sort().to_list()
complaint_mapping = {}
complaint_exprs = []

for i, val in enumerate(complaint_vals):
    key = f"complaint_{i+1:03d}"
    complaint_mapping[key] = val
    complaint_exprs.append(pl.when(pl.col("complaint_type") == val).then(1).otherwise(0).alias(key))

df = df.with_columns(complaint_exprs).drop("complaint_type")

with open(MAPPING_DIR / "complaint_map.json", "w") as f:
    json.dump(complaint_mapping, f, indent=2)

print(f"Complaint Type: {len(complaint_vals)} categories → {len(complaint_exprs)} features")

Complaint Type: 110 categories → 110 features


In [12]:
descriptor_vals = df["descriptor"].unique().sort().to_list()
descriptor_mapping = {}
descriptor_exprs = []
for i, val in enumerate(descriptor_vals):
    key = f"descriptor_{i+1:03d}"
    descriptor_mapping[key] = val
    descriptor_exprs.append(pl.when(pl.col("descriptor") == val).then(1).otherwise(0).alias(key))

df = df.with_columns(descriptor_exprs).drop("descriptor")

with open(MAPPING_DIR / "descriptor_map.json", "w") as f:
    json.dump(descriptor_mapping, f, indent=2)

print(f"Descriptor: {len(descriptor_vals)} categories → {len(descriptor_exprs)} features")

Descriptor: 397 categories → 397 features


In [13]:
location_vals = df["location_type"].unique().sort().to_list()
location_mapping = {}
location_exprs = []

for i, val in enumerate(location_vals):
    key = f"location_{i+1:03d}"
    location_mapping[key] = val
    location_exprs.append(pl.when(pl.col("location_type") == val).then(1).otherwise(0).alias(key))

df = df.with_columns(location_exprs).drop("location_type")

with open(MAPPING_DIR / "location_map.json", "w") as f:
    json.dump(location_mapping, f, indent=2)

print(f"Location Type: {len(location_vals)} categories → {len(location_exprs)} features")

Location Type: 41 categories → 41 features


In [14]:
borough_vals = df["borough"].unique().sort().to_list()
borough_mapping = {}
borough_exprs = []
for i, val in enumerate(borough_vals):
    key = f"borough_{i+1:03d}"
    borough_mapping[key] = val
    borough_exprs.append(pl.when(pl.col("borough") == val).then(1).otherwise(0).alias(key))

df = df.with_columns(borough_exprs).drop("borough")

with open(MAPPING_DIR / "borough_map.json", "w") as f:
    json.dump(borough_mapping, f, indent=2)

print(f"Borough: {len(borough_vals)} categories → {len(borough_exprs)} features")

Borough: 6 categories → 6 features


In [15]:
combo_vals = df["combo"].unique().sort().to_list()
combo_mapping = {}
combo_exprs = []

for i, val in enumerate(combo_vals):
    key = f"combo_{i+1:03d}"
    combo_mapping[key] = val
    combo_exprs.append(pl.when(pl.col("combo") == val).then(1).otherwise(0).alias(key))

df = df.with_columns(combo_exprs).drop("combo")

with open(MAPPING_DIR / "combo_map.json", "w") as f:
    json.dump(combo_mapping, f, indent=2)

print(f"Combo: {len(combo_vals)} categories → {len(combo_exprs)} features")

Combo: 522 categories → 522 features


In [16]:
columns_to_drop = [
    "unique_key",
    "created_date",
    "closed_date",
    "agency_name",
    "resolution_minutes",
    "hour",
    "minute",
    "second",
    "seconds_since_midnight",
    "day_fraction",
    "week_seconds",
    "week_fraction",
    "weekday",
    "latitude",
    "longitude",
]

df = df.drop([c for c in columns_to_drop if c in df.columns])

print(f"Final shape: {df.shape}")
print(f"Final columns ({len(df.columns)}):")
print(df.columns)

Final shape: (1500000, 1098)
Final columns (1098):
['target_bucket', 'time_sin', 'time_cos', 'week_sin', 'week_cos', 'is_weekend', 'latitude_scaled', 'longitude_scaled', 'agency_DCWP', 'agency_DEP', 'agency_DHS', 'agency_DOB', 'agency_DOE', 'agency_DOHMH', 'agency_DOT', 'agency_DPR', 'agency_DSNY', 'agency_HPD', 'agency_NYC311-PRD', 'agency_NYPD', 'agency_OOS', 'agency_TLC', 'complaint_001', 'complaint_002', 'complaint_003', 'complaint_004', 'complaint_005', 'complaint_006', 'complaint_007', 'complaint_008', 'complaint_009', 'complaint_010', 'complaint_011', 'complaint_012', 'complaint_013', 'complaint_014', 'complaint_015', 'complaint_016', 'complaint_017', 'complaint_018', 'complaint_019', 'complaint_020', 'complaint_021', 'complaint_022', 'complaint_023', 'complaint_024', 'complaint_025', 'complaint_026', 'complaint_027', 'complaint_028', 'complaint_029', 'complaint_030', 'complaint_031', 'complaint_032', 'complaint_033', 'complaint_034', 'complaint_035', 'complaint_036', 'complaint

In [17]:
df.write_csv(OUTPUT_PATH)
print(f"Exported {df.shape[0]:,} rows × {df.shape[1]} columns → {OUTPUT_PATH.resolve()}")

Exported 1,500,000 rows × 1098 columns → /Users/aiden/Documents/University/Ciclo 09/Programación Concurrente y Distribuida/TB2-Trabajo-final/data/training/nyc_311_features.csv


In [18]:
print("\n=== Verification ===")
print(f"All numeric: {all(t in [pl.Int64, pl.Float64, pl.Int32, pl.Float32] for t in df.dtypes)}")
print(f"Null count: {df.null_count().sum_horizontal().item()}")
print(f"\nTarget distribution:")
print(df["target_bucket"].value_counts().sort("target_bucket"))


=== Verification ===
All numeric: True
Null count: 0

Target distribution:
shape: (10, 2)
┌───────────────┬────────┐
│ target_bucket ┆ count  │
│ ---           ┆ ---    │
│ i32           ┆ u32    │
╞═══════════════╪════════╡
│ 0             ┆ 178518 │
│ 1             ┆ 161315 │
│ 2             ┆ 102299 │
│ 3             ┆ 163979 │
│ 4             ┆ 125117 │
│ 5             ┆ 187561 │
│ 6             ┆ 131530 │
│ 7             ┆ 147104 │
│ 8             ┆ 164607 │
│ 9             ┆ 137970 │
└───────────────┴────────┘


In [22]:
nonzero_per_row = (
    (df.drop("target_bucket") != 0)
    .sum_horizontal()
    .mean()
)
print(f"Average non-zero features per row (excluding target): {nonzero_per_row:.2f}")

Average non-zero features per row (excluding target): 12.26
